# 01 — Tokenization and inference with RecipeTriage


Start with the setup cell below; the local project package must be available in the active notebook kernel.
Run this notebook with the project Python 3.12 virtual environment. Install `ml[notebook,hf]` as described in LEARNING.md first. First execution downloads Qwen2.5-0.5B-Instruct (roughly 1 GB of weights); the tokenizer is much smaller. No API key is needed here. This notebook performs inference only.

A **pretrained model** learns text patterns through weight updates. An **instruct model** receives additional training to respond to instructions. We use an existing instruct model; we do not train it. [Model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)

The source-linked ravioli summary takes 70 minutes. We will not assume that the word “Easy” changes that time.

## Install the project into the active notebook environment
Run this setup cell before the lesson. It displays the active Python and installs the small local `ml/` package if it is missing. It does not download model weights. Dependency/build-tool installation may require network access; pip errors remain visible.

If you already installed the project using `pip install -e ...` while this notebook was open, restart the kernel first. Editable package registration is normally loaded at interpreter startup. For development, use the editable installation documented in LEARNING.md; the fallback below installs a regular package copy. Reinstall that copy after changing files under `ml/`.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import subprocess
import sys

print("Notebook Python:", sys.executable)
if sys.version_info < (3, 12):
    raise RuntimeError("This project requires Python 3.12+. Select the project Python 3.12 kernel, then restart it.")

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "docker-compose.yml").is_file() and (p / "ml" / "pyproject.toml").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the extracted recipetriage folder, or change the notebook working directory to that folder. The complete ml/ folder is required.")

if importlib.util.find_spec("recipetriage_ml") is None:
    # Install the local project into THIS kernel's Python, not an unrelated terminal Python.
    # A regular install is importable immediately; editable .pth registrations may need a restart.
    subprocess.check_call([sys.executable, "-m", "pip", "install", str(ROOT / "ml")])
    importlib.invalidate_caches()

if importlib.util.find_spec("recipetriage_ml") is None:
    raise RuntimeError("Installation completed but this kernel cannot import the package. Restart the kernel and run the notebook again.")

import recipetriage_ml
print("RecipeTriage package:", recipetriage_ml.__file__)


In [ ]:
import os
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
import torch
from recipetriage_ml.inference.hf_provider import get_tokenizer, get_model, MODEL_ID, MODEL_REVISION
from recipetriage_ml.inference.contracts import Recipe
from recipetriage_ml.inference.experiment import fixture
from recipetriage_ml.inference.prompts import build_messages
from recipetriage_ml.inference.service import triage

tokenizer = get_tokenizer()
recipe = Recipe.model_validate(fixture()["recipe"])
print(MODEL_ID, MODEL_REVISION)
print(recipe.title, recipe.time_minutes, "minutes")

## Tokens and token IDs

A **tokenizer** maps text to vocabulary pieces, called **tokens**, and their integer **token IDs**. IDs are vocabulary addresses, not meaning scores. Different models may use different vocabularies. A word can span several pieces; punctuation and whitespace also affect tokens. Qwen's vocabulary can display byte-level markers such as `Ġ` for a space; decode the entire sequence to reconstruct Unicode text.

In [ ]:
text = "Easy ravioli takes 70 minutes."
ids = tokenizer.encode(text, add_special_tokens=False)
pieces = tokenizer.convert_ids_to_tokens(ids)
for position, (piece, token_id) in enumerate(zip(pieces, ids)):
    print(position, repr(piece), token_id)
print("Count:", len(ids))
assert tokenizer.decode(ids) == text

## Chat template and tensors

A **chat template** arranges message roles, content, separators, and the assistant-generation marker exactly as the model expects. Use the tokenizer's own template. Do not add special tokens a second time after rendering it.

A **tensor** is a numeric array. Our input IDs tensor has shape `[batch_size, input_length]`. The batch size here is one; the attention mask tells the model which positions are real input.

In [ ]:
messages = build_messages(recipe)
rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(rendered)
inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
print("ID tensor shape:", tuple(inputs.input_ids.shape))
print("ID tensor dtype:", inputs.input_ids.dtype)
assert inputs.input_ids[0].tolist() == tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)

## Logits and softmax

For each input position, the model produces **logits**: raw scores over its vocabulary. Use the last position to predict the next token. **Softmax** turns those scores into probabilities that sum to one:

`p_i = exp(z_i / T) / sum_j exp(z_j / T)`

This formula requires `T > 0`. Our application's temperature-zero setting selects greedy decoding instead of dividing by zero. Lower positive temperature concentrates probability; it does not teach the model new facts. Next-token probability is not confidence that a recipe label is true.

We compute the softmax demonstration in float64 to reduce rounding across the large vocabulary. Float32 probabilities can sum slightly above or below one; the model weights remain float32.

In [ ]:
model = get_model()
model.eval()
with torch.inference_mode():
    output = model(**inputs)
logits = output.logits[0, -1, :]
probabilities = torch.softmax(logits.double(), dim=-1)
print("Full logits shape:", tuple(output.logits.shape))
print("Next-token scores shape:", tuple(logits.shape))
print("Probability sum:", probabilities.sum().item())
values, indices = torch.topk(probabilities, 5)
for token_id, probability in zip(indices.tolist(), values.tolist()):
    print(token_id, repr(tokenizer.decode([token_id])), round(probability, 6))
assert torch.isclose(probabilities.sum(), torch.tensor(1.0, dtype=torch.float64), atol=1e-12)

## Temperature, decoding, and max_new_tokens

**Decoding** repeatedly chooses a token, appends it, and computes the next distribution. Greedy decoding chooses the highest score; sampling draws according to probabilities. **Temperature** controls the sampling distribution. **max_new_tokens** caps output length, not total prompt-plus-output length. Generation may stop earlier at an end-of-sequence marker; hitting the cap can truncate JSON.

The next cell compares entropy (uncertainty) for the *same* next-token logits, then runs one complete recipe classification. `model.eval()` changes training-specific layer behavior; `torch.inference_mode()` disables gradient tracking. Neither performs a weight update.

In [ ]:
for temperature in [0.5, 1.0, 1.5]:
    p = torch.softmax(logits.double() / temperature, dim=-1)
    entropy = -(p * p.clamp_min(1e-30).log()).sum().item()
    print("Temperature:", temperature, "entropy:", round(entropy, 4))

# Check that a small sample of weights stays unchanged during generation.
before = next(model.parameters()).detach().flatten()[:32].clone()
result = triage(recipe, "hf", temperature=0.0, max_new_tokens=128)
print(result)
assert torch.equal(before, next(model.parameters()).detach().flatten()[:32])
print("JSON contract valid:", result["valid_json"])
print("Check the labels against 70 minutes yourself.")

## Inference is not training

**Inference** runs the current model to generate tokens. **Training** compares predictions with targets, computes a loss and gradients, and updates weights with an optimizer. There is no backward pass or optimizer step in this notebook.

Our observed small-model run returned schema-valid JSON but incorrectly used `weeknight-30min`. That is a semantic error. The raw title experiment is saved in `ml/experiments/`; all four inputs keep ingredients, instructions, equipment, and time identical.

The Fireworks provider sends the same messages to a hosted model, which applies its own chat template. Its tokenizer and model differ from local Qwen, so this notebook's token counts do not describe Fireworks billing or the hosted model's exact input.

**Your turn:** Explain inference versus training in your own words before continuing to dataset engineering.

References: [Transformers generation](https://huggingface.co/docs/transformers/main_classes/text_generation), [Fireworks chat completions](https://docs.fireworks.ai/api-reference/post-chatcompletions).